# 01 — Coleta e tratamento do RGF

Este notebook executa primeiro uma chamada exploratória para uma UF/ano, inspeciona a estrutura real e depois coleta o último quadrimestre disponível do **Poder Executivo** das 27 UFs entre 2015 e 2025. Ausências não são convertidas em zero.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from rgf.api import exploratory_call, collect_all
from rgf.treatment import build_treated

## 1. Chamada exploratória obrigatória
A API é consultada para São Paulo, 2025, 3º quadrimestre, antes de qualquer mapeamento analítico.

In [ ]:
amostra, estrutura = exploratory_call('SP', 2025, 3)
display(pd.Series(estrutura).to_frame('resultado'))
display(amostra.head())

In [ ]:
alvos = ['ReceitaCorrenteLiquidaLimiteLegal', 'ReceitaCorrenteLiquidaAjustada',
         'DespesaComPessoalTotal', 'LimiteDeAlertaDespesaComPessoalTotal',
         'LimitePrudencialDespesaComPessoalTotal', 'LimiteMaximoDespesaComPessoalTotal']
display(amostra.loc[amostra.cod_conta.isin(alvos), ['cod_conta','conta','coluna','valor']])

## 2. Coleta completa
A API informa limite de uma requisição por segundo; a função aplica pausa, repetição em falhas transitórias, timeout e paginação. A execução leva alguns minutos.

In [ ]:
raw, auditoria_coleta = collect_all()
display(auditoria_coleta.status.value_counts(dropna=False))
display(raw.head())

## 3. Tratamento, mapeamento e qualidade
O mapeamento combina códigos de conta e descrições normalizadas, registrando a origem de cada métrica. O percentual declarado no RGF é preferido; DTP/RCL é calculado somente como fallback.

In [ ]:
base, auditoria_mapeamento = build_treated(raw)
qualidade = {
    'linhas_brutas': len(raw),
    'duplicatas_exatas': int(raw.duplicated().sum()),
    'UF_ano_sem_DTP_RCL': int(base.DTP_RCL.isna().sum()),
    'nomenclaturas_de_conta': int(raw.conta.nunique()),
    'codigos_de_conta': int(raw.cod_conta.nunique()),
}
display(pd.Series(qualidade).to_frame('valor'))
display(pd.crosstab(auditoria_mapeamento.metrica, auditoria_mapeamento.status))
display(base.head())